# Pipeline building
This use-case is pipeline building. We will prepare a MNIST digits dataset for training a classifier.

In [1]:
#!pip3 install torchvision

In [2]:
import torch
import torchvision
import cascade.data as cdd
import torchvision.transforms.functional as F

In [3]:
import cascade
cascade.__version__

'0.18.0'

Let's load torch dataset

In [4]:
MNIST_ROOT = 'data'

train_ds = torchvision.datasets.MNIST(root=MNIST_ROOT,
                                     train=True,
                                     transform=F.to_tensor,
                                     download=True)
test_ds = torchvision.datasets.MNIST(root=MNIST_ROOT,
                                    train=False,
                                    transform=F.to_tensor)

## Creating Cascade Dataset

In [5]:
train_ds = cdd.Wrapper(train_ds)
train_ds.describe("This is MNIST dataset of handwritten images")
test_ds = cdd.Wrapper(test_ds)

## Applying noise
Let's say we want to apply noise to an image.  
*We will use hardcoded magnitude to simplify an example.*

In [6]:
class NoiseModifier(cdd.Modifier):
    def get(self, index):
        img, label = self._dataset[index]
        img += torch.rand_like(img) * 0.1
        img = torch.clip(img, 0, 255)
        return img, label

In [7]:
train_ds = NoiseModifier(train_ds)

## Viewing metadata

In [8]:
from pprint import pprint
pprint(train_ds.get_meta())

[{'comments': [],
  'data_card': None,
  'description': None,
  'len': 60000,
  'links': [],
  'name': '__main__.NoiseModifier',
  'tags': [],
  'type': 'dataset'},
 {'comments': [],
  'data_card': None,
  'description': 'This is MNIST dataset of handwritten images',
  'len': 60000,
  'links': [],
  'name': 'cascade.data.dataset.Wrapper',
  'obj_type': "<class 'torchvision.datasets.mnist.MNIST'>",
  'tags': [],
  'type': 'dataset'}]


## Ready to train model
Now we can set the batch size and pass our pipeline to the DataLoaders.

In [9]:
BATCH_SIZE = 10

In [10]:
trainldr = torch.utils.data.DataLoader(dataset=train_ds,
                                       batch_size=BATCH_SIZE,
                                       shuffle=True)
testldr = torch.utils.data.DataLoader(dataset=test_ds,
                                      batch_size=BATCH_SIZE,
                                      shuffle=False)